# Experimentos de tesis — clasificación del español andino

Este notebook coordina los **cuatro sistemas bajo prueba (SUT)** usando los módulos originales de `src/`. Las predicciones se guardan en Google Drive para no perder resultados si se interrumpe Colab. Puede ejecutarse **en orden, de arriba abajo**.

- **SUT 1:** spaCy `es_core_news_md` + regresión logística, validación cruzada estratificada de cinco folds.
- **SUT 2:** `meta-llama/Llama-3.1-8B-Instruct`, sin recuperación.
- **SUT 3:** Llama + BGE-M3 + FAISS, **top-1 y umbral 0.4**.
- **SUT 4:** Llama + detectores morfosintácticos sobre Stanza, **top-1**.

**Entorno:** Google Colab con GPU T4 para los SUT generativos. Se necesita acceso aprobado al modelo Llama en Hugging Face y un secreto de Colab llamado `HF_TOKEN`.

**Importante:** si ya existen los cuatro CSV completos, el notebook los verifica y **no vuelve a ejecutar inferencias costosas**. Este archivo no recalcula las métricas estadísticas; se hará en el notebook de análisis. El valor 0.4 se adoptó después de una evaluación inicial y debe describirse como ajuste posterior en la metodología.

## 1. Preparar entorno e instalar dependencias
Ejecutar una sola vez por sesión nueva de Colab. La instalación del modelo español de spaCy puede tardar.

In [ ]:
%pip install -q "transformers>=4.46,<5" accelerate bitsandbytes sentence-transformers faiss-cpu stanza spacy scikit-learn scipy pandas numpy matplotlib huggingface_hub

In [ ]:
!python -m spacy download es_core_news_md

## 2. Montar Google Drive y definir rutas

Se utiliza `Mi unidad/tesis_experimento`. Todos los archivos de código y datos permanecen en Drive; el notebook puede estar dentro de `notebooks/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/tesis_experimento")
DATOS = RAIZ / "datos"
RESULTADOS = RAIZ / "resultados"
RESULTADOS.mkdir(parents=True, exist_ok=True)

obligatorios = [
    DATOS / "ejemplos_dialectos_900.csv",
    DATOS / "reglas_andino.txt",
    DATOS / "umbral_rag.txt",
    *[RAIZ / "src" / f for f in (
        "comun.py", "sut1_ml.py", "sut2_llm.py",
        "sut3_rag.py", "sut4_recms.py", "evaluacion.py")],
]
faltantes = [str(p) for p in obligatorios if not p.is_file()]
assert not faltantes, f"Faltan archivos necesarios: {faltantes}"

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Proyecto:", RAIZ)
print("Archivos de código y datos: OK")

## 3. Validar dataset y congelar configuración

**No modificar el umbral después de iniciar una evaluación.** El archivo `datos/umbral_rag.txt` debe contener el valor final `0.4`. Si aparece otro, corregir dicho archivo en Drive antes de continuar.

In [ ]:
import pandas as pd
from src.sut1_ml import cargar_datos
from src.comun import cargar_reglas, MODELO_LLM

DATASET = cargar_datos(DATOS / "ejemplos_dialectos_900.csv")
REGLAS = cargar_reglas(DATOS / "reglas_andino.txt")
UMBRAL_RAG = float((DATOS / "umbral_rag.txt").read_text(encoding="utf-8").strip())

assert len(DATASET) == 900
assert len(REGLAS) == 9
assert set(REGLAS) == {f"R{i:02d}" for i in range(1, 10)}
assert abs(UMBRAL_RAG - 0.4) < 1e-9, (
    f"umbral_rag.txt contiene {UMBRAL_RAG}; para estos resultados debe ser 0.4"
)

print("Frases:", len(DATASET))
print("Distribución:", DATASET["label"].value_counts().to_dict())
print("Reglas:", list(REGLAS))
print("LLM:", MODELO_LLM)
print("Umbral semántico:", UMBRAL_RAG)

## 4. Control de ejecución y validación de archivos

Por defecto, las predicciones existentes se conservan. Para **recalcular desde cero** algún sistema, añade su clave en `REGENERAR`, por ejemplo `{"sut3_rag"}`. Antes de hacer eso, conserva una copia de los resultados y elimina o mueve el respaldo parcial de ese sistema para no mezclar configuraciones.

Los respaldos nuevos se almacenan en `resultados/<sut>_backups_reproducibilidad/`. No se mezclan con los respaldos históricos de la experimentación.

In [ ]:
import os
import time
from pathlib import Path

ARCHIVOS = {
    "sut1_ml": "sut1_ml_predicciones.csv",
    "sut2_llm": "sut2_llm_predicciones.csv",
    "sut3_rag": "sut3_rag_predicciones.csv",
    "sut4_recms": "sut4_recms_predicciones.csv",
}

REGENERAR = set()  # Por defecto, NO sobrescribir resultados existentes
TAMANO_BLOQUE = 25
ETIQUETAS = {"andino", "no_andino"}

def validar_predicciones(df, sut, parcial=False):
    n = len(df)
    if not parcial:
        assert n == 900, f"{sut}: se esperaban 900 filas; hay {n}"
    assert 0 <= n <= 900
    assert df["case_id"].is_unique, f"{sut}: case_id duplicado"
    esperado = DATASET.iloc[:n]
    assert df["case_id"].tolist() == esperado["case_id"].tolist(), f"{sut}: case_id distintos"
    assert df["texto"].tolist() == esperado["test_case"].tolist(), f"{sut}: textos distintos"
    assert df["etiqueta_real"].tolist() == esperado["label"].tolist(), f"{sut}: etiquetas reales distintas"
    assert df["regla_id"].tolist() == esperado["regla_id"].tolist(), f"{sut}: reglas distintas"
    assert df["sut"].eq(sut).all(), f"{sut}: ID de sistema inválido"
    assert df["prediccion"].isin(ETIQUETAS).all(), f"{sut}: predicción inválida"
    if sut == "sut3_rag":
        # Detecta CSV producidos con un umbral distinto.
        hay_regla = df["regla_recuperada"].notna() & df["regla_recuperada"].astype(str).str.strip().ne("")
        deberia_haber_regla = df["score_recuperacion"].ge(UMBRAL_RAG)
        assert (hay_regla == deberia_haber_regla).all(), (
            "SUT 3: el CSV no es coherente con el umbral 0.4; "
            "no mezclarlo con resultados de otros umbrales"
        )
    return df

def necesita_ejecutar(sut):
    ruta = RESULTADOS / ARCHIVOS[sut]
    if sut in REGENERAR:
        return True
    if ruta.exists():
        validar_predicciones(pd.read_csv(ruta), sut)
        print(f"{sut}: 900 predicciones verificadas; se reutilizará {ruta.name}")
        return False
    print(f"{sut}: no hay CSV completo; se generarán predicciones")
    return True

PENDIENTES = {sut: necesita_ejecutar(sut) for sut in ARCHIVOS}
print("Pendientes:", [sut for sut, ejecutar in PENDIENTES.items() if ejecutar] or "ninguno")

## 5. SUT 1 — regresión logística y validación cruzada de cinco folds

Solo se ejecuta si falta el CSV o se incluyó `sut1_ml` en `REGENERAR`. En este SUT no se utiliza GPU.

In [ ]:
if PENDIENTES["sut1_ml"]:
    from src.sut1_ml import cargar_vectorizador, ejecutar_cv
    nlp = cargar_vectorizador()
    predicciones_ml, f1_folds = ejecutar_cv(DATASET, nlp)
    validar_predicciones(predicciones_ml, "sut1_ml")
    ruta = RESULTADOS / ARCHIVOS["sut1_ml"]
    predicciones_ml.to_csv(ruta, index=False, encoding="utf-8-sig")
    print("SUT 1 guardado:", ruta)
else:
    print("SUT 1: resultado existente conservado")

## 6. Preparar Llama solo si se requieren nuevas inferencias

Los SUT 2–4 comparten el **mismo generador**. La autenticación usa el secreto `HF_TOKEN` de Colab; nunca se imprime ni se guarda en el notebook. Es necesario tener acceso autorizado al repositorio del modelo.

In [ ]:
generador = None
if any(PENDIENTES[sut] for sut in ("sut2_llm", "sut3_rag", "sut4_recms")):
    import torch
    from google.colab import userdata
    from huggingface_hub import HfApi
    from src.comun import cargar_llama

    assert torch.cuda.is_available(), (
        "Activa una GPU desde Entorno de ejecución > Cambiar tipo de entorno de ejecución"
    )
    token = userdata.get("HF_TOKEN")
    assert token, "Configura el secreto HF_TOKEN en Colab y autoriza su acceso"
    os.environ["HF_TOKEN"] = token
    HfApi(token=token).whoami()
    print("Autenticación de Hugging Face correcta")
    print("GPU:", torch.cuda.get_device_name(0))
    generador = cargar_llama()
    print("Llama cargado")
else:
    print("No hace falta descargar ni cargar Llama")

## 7. Ejecución reanudable y guardado en Google Drive

Cada bloque de 25 frases se guarda de forma atómica en Drive. Si Colab se desconecta, ejecuta nuevamente las celdas en orden para continuar desde la última fila respaldada. La ejecución se hace sobre el mismo orden del CSV original y se verifica por `case_id`.

La función siguiente no modifica los algoritmos de los módulos `src/`.

In [ ]:
def ejecutar_en_bloques(sut, funcion_prediccion):
    assert sut in {"sut2_llm", "sut3_rag", "sut4_recms"}
    carpeta = RESULTADOS / f"{sut}_backups_reproducibilidad"
    carpeta.mkdir(parents=True, exist_ok=True)
    respaldo = carpeta / f"{sut}_parcial.csv"
    destino_final = RESULTADOS / ARCHIVOS[sut]

    if respaldo.exists():
        acumuladas = pd.read_csv(respaldo)
        validar_predicciones(acumuladas, sut, parcial=True)
    else:
        acumuladas = pd.DataFrame()

    n = len(acumuladas)
    print(f"{sut}: {n}/900 casos ya guardados")

    for inicio in range(n, len(DATASET), TAMANO_BLOQUE):
        fin = min(inicio + TAMANO_BLOQUE, len(DATASET))
        bloque = DATASET.iloc[inicio:fin].copy()
        t0 = time.perf_counter()
        nuevas = funcion_prediccion(bloque)

        assert len(nuevas) == len(bloque)
        assert nuevas["case_id"].tolist() == bloque["case_id"].tolist()
        assert nuevas["prediccion"].isin(ETIQUETAS).all()

        acumuladas = pd.concat([acumuladas, nuevas], ignore_index=True)
        validar_predicciones(acumuladas, sut, parcial=True)

        temporal = respaldo.with_suffix(".tmp")
        acumuladas.to_csv(temporal, index=False)
        os.replace(temporal, respaldo)
        print(f"{sut}: {len(acumuladas)}/900 | bloque {time.perf_counter()-t0:.1f}s", flush=True)

    validar_predicciones(acumuladas, sut)
    temporal_final = destino_final.with_suffix(".tmp")
    acumuladas.to_csv(temporal_final, index=False)
    os.replace(temporal_final, destino_final)
    print(f"{sut}: CSV definitivo guardado en {destino_final}")
    return destino_final

## 8. SUT 2 — LLM nativo (sin RAG)

In [ ]:
if PENDIENTES["sut2_llm"]:
    from src.sut2_llm import predecir_dataset as predecir_sut2
    ejecutar_en_bloques(
        "sut2_llm",
        lambda bloque: predecir_sut2(bloque, generador)
    )
else:
    print("SUT 2: resultado existente conservado")

## 9. SUT 3 — RAG semántico BGE-M3 / FAISS

Se codifican nueve reglas completas, se recupera como máximo una regla y se aplica la similitud coseno con umbral **0.4**. El embedding y FAISS se ejecutan en CPU.

In [ ]:
if PENDIENTES["sut3_rag"]:
    from src.sut3_rag import (
        cargar_recuperador as cargar_rag,
        predecir_dataset as predecir_sut3,
    )
    recuperador_rag = cargar_rag(
        umbral_similitud=UMBRAL_RAG,
        ruta_reglas=DATOS / "reglas_andino.txt",
        dispositivo="cpu",
    )
    assert recuperador_rag["indice"].ntotal == 9
    ejecutar_en_bloques(
        "sut3_rag",
        lambda bloque: predecir_sut3(bloque, generador, recuperador_rag)
    )
else:
    print("SUT 3: resultado existente conservado")

## 10. SUT 4 — recuperador morfosintáctico Stanza

Stanza procesa tokenización, palabras multitoken, categorías gramaticales, lemas y dependencias en **CPU**. Los nueve detectores conservan el orden de prioridad de `src/sut4_recms.py`.

In [ ]:
if PENDIENTES["sut4_recms"]:
    import stanza
    from src.sut4_recms import (
        cargar_recuperador as cargar_recms,
        predecir_dataset as predecir_sut4,
    )
    stanza.download(
        "es", processors="tokenize,mwt,pos,lemma,depparse", verbose=False
    )
    recuperador_recms = cargar_recms(ruta_reglas=DATOS / "reglas_andino.txt")
    assert len(recuperador_recms["reglas"]) == 9
    ejecutar_en_bloques(
        "sut4_recms",
        lambda bloque: predecir_sut4(bloque, generador, recuperador_recms)
    )
else:
    print("SUT 4: resultado existente conservado")

## 11. Verificación final de los cuatro SUT

Esta celda garantiza que están las **900 predicciones** de cada sistema y que corresponden a los mismos casos y etiquetas. Las métricas y figuras se generan desde estos CSV en `analisis_resultados.ipynb`.

In [ ]:
print("=== COMPROBACIÓN FINAL ===")
for sut, archivo in ARCHIVOS.items():
    df = pd.read_csv(RESULTADOS / archivo)
    validar_predicciones(df, sut)
    n_aciertos = int(df["prediccion"].eq(df["etiqueta_real"]).sum())
    print(f"{sut}: 900/900 | aciertos: {n_aciertos} | {archivo}")
print("TOTAL: 3600 predicciones verificadas")